# Valuing a CLP swap under USD collateral

This notebook values a five-year swap that receives 5.5% fixed and pays ICP
floating on 935 million CLP. The trade is collateralized in USD. We value it
in USD, then rebuild that value from its CLP cashflows.

Three market inputs drive the value:

- the ICP curve, which projects the floating CLP coupons;
- a CLP discount curve that reflects USD collateral;
- the USD/CLP spot exchange rate.

The [local loader](market_data.py) reads an illustrative USD/CLP market dated
24 February 2026. Spot is 935 CLP per USD, so the notional is worth USD 1
million on the start date. Run the cells in order after the
[shared setup](../../README.md#install-and-run).


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_currency_market

market = load_currency_market()
ref = market.quotes.reference_date
print("Market reference date:", ref)
pd.DataFrame(
    {"configured_curve": [curve.to_dict()["market_index"] for curve in market.curves]}
)


Market reference date: 2026-02-24


,configured_curve
0,SOFR
1,"{'Collateral': ['CLP', 'USD']}"
2,ICP


## FX orientation

`FxStore.add(base, quote, rate)` stores the number of quote-currency units per
one unit of the base currency. We enter USD/CLP spot as 935: one USD buys 935
CLP. Asking for the reverse rate returns `1 / 935`, so 9,350,000 CLP converts
to 10,000 USD.

The base currency of an FX pair is not the currency the valuation reports in.
Here the collateral (CSA) policy makes USD the reporting currency, while both
legs of the swap pay CLP.


In [2]:
fx = qs.FxStore()
fx.add(qs.Currency.USD, qs.Currency.CLP, 935.0)

clp_to_usd = fx.rate(qs.Currency.CLP, qs.Currency.USD)
usd_to_clp = fx.rate(qs.Currency.USD, qs.Currency.CLP)

pd.DataFrame(
    [
        {
            "from": "CLP",
            "to": "USD",
            "amount": 9_350_000,
            "fx_rate": clp_to_usd,
            "converted_amount": 9_350_000 * clp_to_usd,
        },
        {
            "from": "USD",
            "to": "CLP",
            "amount": 10_000,
            "fx_rate": usd_to_clp,
            "converted_amount": 10_000 * usd_to_clp,
        },
        {
            "from": "USD",
            "to": "USD",
            "amount": 10_000,
            "fx_rate": fx.rate(qs.Currency.USD, qs.Currency.USD),
            "converted_amount": 10_000 * fx.rate(qs.Currency.USD, qs.Currency.USD),
        },
    ]
)


,from,to,amount,fx_rate,converted_amount
0,CLP,USD,9350000,0.00107,10000.0
1,USD,CLP,10000,935.00000,9350000.0
2,USD,USD,10000,1.00000,10000.0


## Keep projection, collateral, and reporting roles explicit

We receive a 5.5% fixed coupon and pay ICP floating coupons for five years.
The notional is 935 million CLP, which is USD 1 million at today's spot. Both
legs pay CLP, and their principal exchanges cancel. The USD collateral
agreement pays interest at SOFR and makes USD the reporting currency of the
pricing context.

| Market object | Role |
| --- | --- |
| `ICP` curve | Projects the CLP floating coupons |
| `SOFR` curve | Gives the interest paid on USD collateral; the CLP discount curve is built from it |
| `Collateral(CLP, USD)` curve | Discounts CLP cashflows under USD collateral |
| `FxStore` | Converts CLP amounts to USD at spot |

The CLP discount curve under USD collateral is fitted to FX-forward and
cross-currency swap quotes, so it includes the cross-currency basis.
Discounting with the ICP curve instead would value the swap as if it had
different collateral.


In [3]:
collateral_index = qs.MarketIndex.collateral(qs.Currency.CLP, qs.Currency.USD)
swap = qs.Swap(
    identifier="CLP-SWAP-USD-CSA-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=935_000_000,
    fixed_rate=0.055,
    currency=qs.Currency.CLP,
    market_index=qs.MarketIndex.ICP,
    fixed_leg_frequency=qs.Frequency.Semiannual,
    floating_leg_frequency=qs.Frequency.Semiannual,
)
trade = qs.Trade(swap, ref, qs.Side.LongReceive)


## Let the pricer convert to the reporting currency

Pass both the FX store and the USD collateral policy to `PricingContext`.
Inside `with context:`, the valuation builds the curves it needs. Because of
the policy, `result.price` is a **USD PV**. The cashflow table
`result.cashflows` keeps the payment amounts in **CLP**.

The pricer computes

$$
PV_{USD} = \sum_i s_i\,CF_{CLP,i}\,S_{CLP\rightarrow USD}\,
P_{CLP\mid USD}(T_i).
$$

Here $CF_{CLP,i}$ is a CLP payment on date $T_i$, $S_{CLP\rightarrow USD}$ is
the spot rate in USD per CLP, and $P_{CLP\mid USD}(T_i)$ is the CLP discount
factor under USD collateral. The sign $s_i$ is +1 for the fixed leg we receive
and −1 for the floating leg we pay.

The cashflow table includes these signs, and it gives the initial principal
payment the opposite sign to the repayment. For a `Swap`, leg 0 is fixed and
leg 1 is floating. The next cell applies the discount factors and the FX rate
to the table. Curve queries must run inside the `with` block; the resulting
table can be used after it.


In [4]:
context = qs.PricingContext(
    quotes=market.quotes, curves=market.curves, fx=fx, csa_policy=market.csa_policy
)

with context:
    result = context.evaluate(trade, [qs.Request.Value, qs.Request.Cashflows()])
    base_currency = context.base_currency
    conversion_rate = context.fx.rate(swap.currency, base_currency)
    discount_curve = context.curve(collateral_index)
    cashflows = result.cashflows.copy()
    cashflows["leg"] = cashflows["leg_index"].map(
        {0: "receive fixed", 1: "pay floating"}
    )
    cashflows["signed_amount_clp"] = cashflows["amount"]
    cashflows["discount_factor"] = cashflows["payment_date"].map(
        discount_curve.discount_factor
    )
    cashflows["pv_clp"] = cashflows["signed_amount_clp"] * cashflows["discount_factor"]
    cashflows["fx_clp_to_usd"] = conversion_rate
    cashflows["pv_usd"] = cashflows["pv_clp"] * conversion_rate

cashflows.sort_values(["payment_date", "leg_index"])[
    [
        "payment_date",
        "leg",
        "currency",
        "signed_amount_clp",
        "discount_factor",
        "fx_clp_to_usd",
        "pv_usd",
    ]
].head(8)


,payment_date,leg,currency,signed_amount_clp,discount_factor,fx_clp_to_usd,pv_usd
0,2026-02-24,receive fixed,CLP,-9.350000e+08,1.000000,0.00107,-1000000.000000
12,2026-02-24,pay floating,CLP,9.350000e+08,1.000000,0.00107,1000000.000000
2,2026-08-24,receive fixed,CLP,2.585535e+07,0.963357,0.00107,26639.504550
14,2026-08-24,pay floating,CLP,-2.327727e+07,0.963357,0.00107,-23983.239411
3,2027-02-24,receive fixed,CLP,2.628389e+07,0.950521,0.00107,26720.201442
15,2027-02-24,pay floating,CLP,-2.216951e+07,0.950521,0.00107,-22537.521210
4,2027-08-24,receive fixed,CLP,2.585535e+07,0.929084,0.00107,25691.763794
16,2027-08-24,pay floating,CLP,-2.112135e+07,0.929084,0.00107,-20987.713675


## Reconcile the position PV in both units

The sum of the signed, discounted payments is the position's value in CLP,
still under USD collateral. Multiplying it by the CLP-to-USD spot rate gives
the library's USD value. Both numbers describe the same collateral agreement
in two currencies. A swap collateralized in CLP would need a different
discount curve.


In [5]:
local_pv_clp = cashflows["pv_clp"].sum()
reconstructed_pv_usd = cashflows["pv_usd"].sum()

pd.DataFrame(
    [
        {
            "measure": "PV expressed in CLP under USD collateral",
            "currency": "CLP",
            "value": local_pv_clp,
        },
        {
            "measure": "Converted cashflow PV",
            "currency": "USD",
            "value": reconstructed_pv_usd,
        },
        {
            "measure": "Native result.price",
            "currency": str(base_currency),
            "value": result.price,
        },
        {
            "measure": "Reconciliation difference",
            "currency": "USD",
            "value": reconstructed_pv_usd - result.price,
        },
    ]
)


,measure,currency,value
0,PV expressed in CLP under USD collateral,CLP,4.790493e+07
1,Converted cashflow PV,USD,5.123522e+04
2,Native result.price,USD,5.123522e+04
3,Reconciliation difference,USD,-7.275958e-12


## Principal exchanges

The fixed and floating legs exchange the same principal with opposite signs,
so the amounts cancel on each date. The cell below checks this and computes
the PV of the coupons alone.


In [6]:
principal_flows = cashflows.loc[cashflows["type"].isin(["Disbursement", "Redemption"])]
principal_by_date = principal_flows.groupby("payment_date")["signed_amount_clp"].sum()
coupon_pv_usd = cashflows.loc[
    ~cashflows["type"].isin(["Disbursement", "Redemption"]), "pv_usd"
].sum()
print(f"USD-collateralized PV: CLP {local_pv_clp:,.2f} = USD {result.price:,.2f}.")
print(f"At {usd_to_clp:g} CLP per USD, one CLP is worth USD {conversion_rate:.9f}.")
print(f"Net principal PV: USD {principal_flows['pv_usd'].sum():.3e}.")


USD-collateralized PV: CLP 47,904,926.39 = USD 51,235.22.
At 935 CLP per USD, one CLP is worth USD 0.001069519.
Net principal PV: USD 0.000e+00.


`result.price` is in USD, so use it directly in a USD portfolio report. Do not
apply the CLP-to-USD rate to it, because that would convert it twice. When you
show the contract cashflows, keep their CLP labels and add separate columns
for the USD values.

Continue with [portfolio valuation and risk](../../sensitivity/portfolio_risk.ipynb)
for aggregation, or see the [valuation catalog](../README.md) for the results
available on other contracts.